# E1 — LSTM Baseline for RUL Prediction on C-MAPSS FD001

**Study:** Agentic Multi-Machine Predictive Maintenance (AMMPM) using Time-Series Foundation Models for Explainable and Trustworthy RUL Prediction.

**Purpose:** Establish a classical sequence-model baseline (2-layer LSTM) for Remaining Useful Life (RUL) prediction on the single-operating-condition, single-fault-mode C-MAPSS subset (FD001), before benchmarking richer architectures (GRU, TCN, Transformer, PatchTST, Chronos fine-tune — see `configs/experiment_config.yaml::models`) and the agentic multi-machine orchestration layer in later experiments (E2-E8). Every metric reported here anchors the improvement claims of subsequent experiments.

**Reproducibility contract:** global seed fixed to 42 via `src/utils/seed.py::set_global_seed` (called before any data loading or model construction), CPU-only execution for bit-exact determinism, and a dedicated seeded generator for minibatch shuffling. Restart the kernel and *Run All* to reproduce every number in this notebook exactly.


## 0. Setup

Import shared project utilities from `src/` and `configs/` (not re-implemented here, so every experiment notebook in the AMMPM suite stays consistent), then fix the global seed as the very first executable step.


In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset


def _find_project_root(start: Path) -> Path:
    """Walk upward from `start` until a directory containing configs/paths.py is found."""
    for candidate in (start, *start.parents):
        if (candidate / "configs" / "paths.py").exists():
            return candidate
    raise RuntimeError(
        "Could not locate the AMMPM project root (no configs/paths.py found above "
        f"{start}). Launch Jupyter from the project root or notebooks/ directory."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs.paths import RESULTS_DIR, ensure_dirs  # noqa: E402
from src.data.cmapss_loader import get_cmapss  # noqa: E402
from src.utils.seed import set_global_seed  # noqa: E402

SEED = 42
set_global_seed(SEED)
ensure_dirs()

print(f"Project root: {PROJECT_ROOT}")
print(f"Global seed:  {SEED}")


Project root: /home/bruce-wayne-2005/industrial-ai-project
Global seed:  42


## 1. Data: NASA C-MAPSS FD001

C-MAPSS FD001 simulates a fleet of 100 turbofan engines run to failure under a single operating condition with a single fault mode (HPC degradation) — the easiest of the four C-MAPSS subsets, and the standard starting point for a new RUL model family in the literature. `src/data/cmapss_loader.get_cmapss` handles acquisition, min-max normalization (fit on the training split only, to avoid test-set leakage), and RUL capping at 125 cycles: early in an engine's life degradation has not yet begun, so the true RUL is not learnable from sensor readings alone, and papers since Heimes (2008) / Saxena (2008) cap the label at a plateau to keep the regression target learnable.


In [2]:
DATA = get_cmapss(fd_num=1, max_rul=125)
train_df = DATA["train_df"]
test_df = DATA["test_df"]
feature_columns = DATA["feature_columns"]

print(f"FD001 train: {train_df['unit_number'].nunique()} engines, {len(train_df)} rows")
print(f"FD001 test:  {test_df['unit_number'].nunique()} engines, {len(test_df)} rows")
print(f"Feature channels ({len(feature_columns)}): {feature_columns}")


FD001 train: 100 engines, 20631 rows
FD001 test:  100 engines, 13096 rows
Feature channels (24): ['op_setting_1', 'op_setting_2', 'op_setting_3', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21']


## 2. From per-cycle readings to fixed-length windows

An LSTM consumes a *sequence* of consecutive cycles, not a single row. We slide a window of `SEQUENCE_LENGTH = 30` cycles across each engine's trajectory. Thirty is chosen because it is at or below the shortest trajectory in **both** splits (min 128 cycles in train, min 31 in test), so every engine yields a genuine full-length window with no synthetic padding needed in practice — the padding branch below exists only as a defensive fallback for other C-MAPSS subsets (FD002-FD004) that this windowing function will be reused for in later experiments.

For **training** we take every valid window per engine — a standard, leakage-free augmentation for C-MAPSS (overlapping sub-trajectories of the same run-to-failure curve). For **test** we take only the final window per engine, matching the official PHM08 scoring protocol of one RUL prediction per engine at its last reported cycle.


In [3]:
SEQUENCE_LENGTH = 30  # cycles per input window


def build_windows(df: pd.DataFrame, feature_cols, window: int, mode: str):
    """Slice per-engine trajectories into fixed-length sliding windows.

    mode="train": every valid window per engine (overlapping sub-trajectory
        augmentation, standard for C-MAPSS LSTM baselines).
    mode="last": only the final window per engine (one prediction per test
        engine, matching the official PHM08 scoring protocol).
    """
    X_list, y_list = [], []
    for _, group in df.groupby("unit_number"):
        group = group.sort_values("time_in_cycles")
        feats = group[feature_cols].to_numpy(dtype=np.float32)
        rul = group["RUL"].to_numpy(dtype=np.float32)
        n = len(group)

        if n < window:
            # Left-pad short trajectories by repeating the earliest reading,
            # so every engine yields at least one full-length window.
            pad = np.repeat(feats[:1], window - n, axis=0)
            feats = np.concatenate([pad, feats], axis=0)
            rul = np.concatenate([np.repeat(rul[:1], window - n), rul])
            n = window

        if mode == "last":
            X_list.append(feats[-window:])
            y_list.append(rul[-1])
        else:
            for end in range(window, n + 1):
                X_list.append(feats[end - window:end])
                y_list.append(rul[end - 1])

    return np.stack(X_list).astype(np.float32), np.array(y_list, dtype=np.float32)


## 3. Train/validation split — by engine, not by row

Splitting *rows* 80/20 would leak information: overlapping windows from the same engine trajectory would land on both sides of the split, letting validation loss measure memorization of a trajectory the model has already partially seen rather than generalization to an unseen engine. We instead split the 100 training **engine units** 80/20 and only then generate windows within each split, so validation engines are fully held out from training.


In [4]:
all_units = sorted(train_df["unit_number"].unique())
train_units, val_units = train_test_split(
    all_units, test_size=0.2, random_state=SEED, shuffle=True
)

train_split_df = train_df[train_df["unit_number"].isin(train_units)]
val_split_df = train_df[train_df["unit_number"].isin(val_units)]

X_train, y_train = build_windows(train_split_df, feature_columns, SEQUENCE_LENGTH, mode="train")
X_val, y_val = build_windows(val_split_df, feature_columns, SEQUENCE_LENGTH, mode="train")
X_test, y_test = build_windows(test_df, feature_columns, SEQUENCE_LENGTH, mode="last")

print(f"Train engines: {len(train_units)}  -> {X_train.shape[0]} windows")
print(f"Val engines:   {len(val_units)}  -> {X_val.shape[0]} windows")
print(f"Test engines:  {X_test.shape[0]}  -> {X_test.shape[0]} windows (one per engine)")
print(f"Window shape:  (sequence_length={X_train.shape[1]}, n_features={X_train.shape[2]})")


Train engines: 80  -> 14241 windows
Val engines:   20  -> 3490 windows
Test engines:  100  -> 100 windows (one per engine)
Window shape:  (sequence_length=30, n_features=24)


In [5]:
device = torch.device("cpu")  # forced for bit-exact reproducibility (see src/utils/seed.py)


def to_tensor_dataset(X, y):
    return TensorDataset(torch.from_numpy(X), torch.from_numpy(y).unsqueeze(1))


train_dataset = to_tensor_dataset(X_train, y_train)
val_dataset = to_tensor_dataset(X_val, y_val)
test_dataset = to_tensor_dataset(X_test, y_test)

BATCH_SIZE = 256
# Dedicated, explicitly-seeded generator for minibatch shuffling: decouples
# batch order from whatever else has drawn on the global torch RNG earlier
# in the notebook, so results stay identical even if cells above are edited.
shuffle_generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    generator=shuffle_generator, num_workers=0,
)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Device: {device}")
print(f"Batches per epoch (train): {len(train_loader)}")


Device: cpu
Batches per epoch (train): 56


## 4. Model: 2-layer LSTM baseline

A minimal recurrent baseline: two stacked LSTM layers (hidden size 64) with inter-layer dropout (0.2) for regularization, reading a 30-cycle window of all 24 operational-setting/sensor channels and regressing a single scalar RUL from the final time step's hidden state. No attention, no foundation-model pretraining — this is the floor every later architecture in the AMMPM benchmark (`configs/experiment_config.yaml::models`) must clear.


In [6]:
class LSTMRULRegressor(nn.Module):
    """2-layer LSTM regressor mapping a sensor-window to a scalar RUL."""

    def __init__(self, input_size: int, hidden_size: int = 64, num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_step = out[:, -1, :]
        return self.head(last_step)


HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.2

model = LSTMRULRegressor(
    input_size=len(feature_columns),
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(device)

model


LSTMRULRegressor(
  (lstm): LSTM(24, 64, num_layers=2, batch_first=True, dropout=0.2)
  (head): Linear(in_features=64, out_features=1, bias=True)
)

## 5. Training configuration

Adam (lr = 1e-3), MSE loss on raw RUL cycles, batch size 256, 50 epochs, no early stopping or learning-rate scheduling — kept deliberately simple so this baseline isolates the effect of *architecture* in later experiments rather than the effect of training tricks.


In [7]:
EPOCHS = 50
LEARNING_RATE = 1e-3

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, n_train_examples = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
        n_train_examples += xb.size(0)
    train_epoch_loss = running_loss / n_train_examples

    model.eval()
    running_val_loss, n_val_examples = 0.0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            running_val_loss += loss.item() * xb.size(0)
            n_val_examples += xb.size(0)
    val_epoch_loss = running_val_loss / n_val_examples

    train_losses.append(train_epoch_loss)
    val_losses.append(val_epoch_loss)

    if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:
        print(f"Epoch {epoch:3d}/{EPOCHS} | train MSE: {train_epoch_loss:8.3f} | val MSE: {val_epoch_loss:8.3f}")


Epoch   1/50 | train MSE: 7729.344 | val MSE: 7063.294


Epoch  10/50 | train MSE: 3790.454 | val MSE: 3581.170


Epoch  20/50 | train MSE: 2226.737 | val MSE: 2145.157


Epoch  30/50 | train MSE: 1813.778 | val MSE: 1788.832


Epoch  40/50 | train MSE: 1752.249 | val MSE: 1744.370


Epoch  50/50 | train MSE: 1748.144 | val MSE: 1743.737


## 6. Training/validation loss curve

A diverging train/val curve here would flag overfitting to specific engine trajectories before it ever reaches the test-set metrics below — worth checking before trusting any downstream RMSE/MAE/PHM number.


In [8]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, EPOCHS + 1), train_losses, label="Train MSE")
plt.plot(range(1, EPOCHS + 1), val_losses, label="Validation MSE")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss (RUL, cycles$^2$)")
plt.title("E1 LSTM Baseline — Training/Validation Loss (C-MAPSS FD001)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "E1_lstm_fd001_loss_curve.png", dpi=150)
plt.show()


<Figure size 800x500 with 1 Axes>

## 7. Evaluation: RMSE, MAE, and the PHM08 asymmetric score

RMSE and MAE summarize point-prediction accuracy but treat early and late errors symmetrically — inappropriate for maintenance scheduling, where a *late* prediction (the model claims an engine has more life left than it does) risks an in-service failure, while an *early* prediction only costs a conservative maintenance window. The PHM08 challenge score encodes this asymmetry directly: for error `d = predicted - actual`, early errors (`d < 0`) cost `exp(-d/13) - 1`, late errors (`d >= 0`) cost the steeper `exp(d/10) - 1`. We report all three so later experiments (E2+) can be judged on both raw accuracy and operational risk.

Predictions are clipped at zero before scoring — a plant engineer reading a gauge would never act on a negative "remaining life."


In [9]:
def phm_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """PHM08 challenge asymmetric scoring function (lower is better)."""
    d = y_pred - y_true
    scores = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return float(np.sum(scores))


model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        pred = model(xb)
        all_preds.append(pred.cpu().numpy())
        all_targets.append(yb.numpy())

y_pred = np.concatenate(all_preds).flatten()
y_true = np.concatenate(all_targets).flatten()
y_pred_clipped = np.clip(y_pred, a_min=0.0, a_max=None)

rmse = float(np.sqrt(np.mean((y_pred_clipped - y_true) ** 2)))
mae = float(np.mean(np.abs(y_pred_clipped - y_true)))
phm = phm_score(y_true, y_pred_clipped)

print(f"Test RMSE:      {rmse:.3f} cycles")
print(f"Test MAE:       {mae:.3f} cycles")
print(f"Test PHM Score: {phm:.3f}  (n={len(y_true)} engines)")


Test RMSE:      40.532 cycles
Test MAE:       35.097 cycles
Test PHM Score: 18182.324  (n=100 engines)


## 8. Persisting results

Metrics are written to `results/E1_lstm_fd001.json` alongside the hyperparameters that produced them, so this run can be diffed against E2+ (GRU/TCN/Transformer/PatchTST/Chronos) in the eventual cross-experiment comparison report without re-reading this notebook.


In [10]:
metrics = {
    "experiment_id": "E1",
    "model": "lstm_baseline",
    "dataset": "cmapss",
    "fd_subset": "FD001",
    "seed": SEED,
    "hyperparameters": {
        "sequence_length": SEQUENCE_LENGTH,
        "hidden_size": HIDDEN_SIZE,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS,
        "max_rul": 125,
    },
    "metrics": {
        "rmse": rmse,
        "mae": mae,
        "phm_score": phm,
    },
    "n_test_engines": int(len(y_true)),
}

results_path = RESULTS_DIR / "E1_lstm_fd001.json"
with open(results_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved metrics to {results_path}")
metrics


Saved metrics to /home/bruce-wayne-2005/industrial-ai-project/results/E1_lstm_fd001.json


{'experiment_id': 'E1',
 'model': 'lstm_baseline',
 'dataset': 'cmapss',
 'fd_subset': 'FD001',
 'seed': 42,
 'hyperparameters': {'sequence_length': 30,
  'hidden_size': 64,
  'num_layers': 2,
  'dropout': 0.2,
  'batch_size': 256,
  'learning_rate': 0.001,
  'epochs': 50,
  'max_rul': 125},
 'metrics': {'rmse': 40.532047271728516,
  'mae': 35.096893310546875,
  'phm_score': 18182.32421875},
 'n_test_engines': 100}

## Persisting per-engine predictions

Saved alongside the aggregate metrics so downstream analysis/visualization notebooks (e.g. `EX_visualizations.ipynb`) can read per-engine true/predicted RUL directly, instead of re-running training.


In [11]:
predictions_path = RESULTS_DIR / "E1_lstm_fd001_predictions.npz"
np.savez(predictions_path, y_true=y_true, y_pred=y_pred_clipped)

print(f"Saved per-engine predictions to {predictions_path}")


Saved per-engine predictions to /home/bruce-wayne-2005/industrial-ai-project/results/E1_lstm_fd001_predictions.npz


## Summary

This baseline's RMSE / MAE / PHM score become the reference point for every subsequent experiment in the AMMPM study — GRU/TCN/Transformer comparisons, PatchTST and Chronos fine-tuning, and eventually the agentic multi-machine orchestration and SHAP-based explainability layers (`configs/experiment_config.yaml::experiment_ids`). Because the seed, split, and windowing are fixed and CPU execution is forced for determinism, re-running this notebook end-to-end (kernel restart + *Run All*) should reproduce every number above exactly.
